# 最终候选方案 official test 评估

本 notebook 固定 validation 阶段选出的结构特征方案和阈值，在 official test 上进行结果观察。该结果是 README 中 final candidate 的来源。最终候选为 `median_all_structural_all / structural_all`，阈值为 `0.18`。

## 1. 固定候选策略与阈值

本节读取 Day13 validation 阶段筛选出的候选结构特征方案，并固定阈值进入 official test 观察。策略和 threshold 来自 validation 阶段，official test 只做固定评估。

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd

from scania_aps.config import get_config
from scania_aps.data.load_data import load_train_test_with_target
from scania_aps.data.split_data import split_train_valid
from scania_aps.features.structural_feature_design import load_structural_feature_config
from scania_aps.models.structural_feature_test_evaluation import (
    DEFAULT_DAY14_CANDIDATE_GROUPS,
    evaluate_structural_feature_candidates_on_test,
)

cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")
structural_config = load_structural_feature_config(PROJECT_ROOT / "config" / "structural_features.yaml")
valid_best_summary = pd.read_csv(cfg.metrics_dir / "day13_structural_feature_valid_best_summary.csv")

候选组固定为 baseline、selected missing indicators、prefix zero rate 和 structural_all。候选集合来自 validation 阶段设计，不根据 test 表现再增删。

In [2]:
candidate_groups = DEFAULT_DAY14_CANDIDATE_GROUPS
valid_best_summary[valid_best_summary["strategy"].isin(candidate_groups)][[
    "strategy", "best_threshold", "precision", "recall", "f2", "average_precision", "fp", "fn", "total_cost"
]]

,strategy,best_threshold,precision,recall,f2,average_precision,fp,fn,total_cost
0,median_all_structural_all,0.18,0.367925,0.975,0.733083,0.863543,335,5,5850
1,median_all_selected_missing_indicators_top30,0.30,0.430804,0.965,0.773237,0.860375,255,7,6050
2,median_all_prefix_zero_rate,0.09,0.293592,0.985,0.669613,0.862527,474,3,6240
4,baseline_median_all,0.16,0.359259,0.970,0.723881,0.867239,346,6,6460


## 2. official test 结果

本节在固定候选策略和阈值下评估 official test。模型训练仍基于 official train 内部数据，test 只用于 transform 和结果观察。

In [3]:
train_df, test_df = load_train_test_with_target(cfg)
train_inner_df, valid_df = split_train_valid(train_df, cfg)

results = evaluate_structural_feature_candidates_on_test(
    train_inner_df=train_inner_df,
    valid_df=valid_df,
    test_df=test_df,
    cfg=cfg,
    structural_config=structural_config,
    candidate_groups=candidate_groups,
    valid_best_summary=valid_best_summary,
)

test_results = results["test_results"]
test_predictions = results["test_predictions"]
valid_test_compare = results["test_compare_with_valid"]
metadata = results["metadata"]

下方结果表重点复核 total_cost、FN、Recall、F2 和 PR-AUC。该任务的核心不是 accuracy，而是控制漏检成本。

In [4]:
display_cols = [
    "candidate_group", "threshold", "precision", "recall", "f2",
    "average_precision", "fp", "fn", "total_cost", "n_structural_features"
]
test_results[display_cols]

,candidate_group,threshold,precision,recall,f2,average_precision,fp,fn,total_cost,n_structural_features
0,median_all_structural_all,0.18,0.477004,0.968000,0.802742,0.905455,398,12,9980,60
1,baseline_median_all,0.16,0.455919,0.965333,0.789015,0.908623,432,13,10820,0
2,median_all_prefix_zero_rate,0.09,0.384615,0.973333,0.745202,0.909259,584,10,10840,5
3,median_all_selected_missing_indicators_top30,0.30,0.545736,0.938667,0.820513,0.908459,293,23,14430,30


结果显示，final candidate 的 official test 结果为 TP=`363`、FP=`398`、TN=`15227`、FN=`12`，total_cost=`9980`。FN 控制是该任务的关键。

## 3. 与 baseline 对比

valid/test 对比表用于确认结构特征方案是否相对 baseline 保持泛化收益。

In [5]:
valid_test_compare

,candidate_group,threshold,valid_precision,valid_recall,valid_f1,valid_f2,valid_average_precision,valid_fp,valid_fn,valid_total_cost,...,test_average_precision,test_fp,test_fn,test_total_cost,delta_total_cost_test_minus_valid,delta_fn_test_minus_valid,delta_fp_test_minus_valid,delta_recall_test_minus_valid,delta_f2_test_minus_valid,note
0,median_all_structural_all,0.18,0.367925,0.975,0.534247,0.733083,0.863543,335,5,5850,...,0.905455,398,12,9980,4130,7,63,-0.007000,0.069659,test 仅用于最终观察；不允许根据该表反向修改候选组或阈值
1,baseline_median_all,0.16,0.359259,0.970,0.524324,0.723881,0.867239,346,6,6460,...,0.908623,432,13,10820,4360,7,86,-0.004667,0.065134,test 仅用于最终观察；不允许根据该表反向修改候选组或阈值
2,median_all_prefix_zero_rate,0.09,0.293592,0.985,0.452354,0.669613,0.862527,474,3,6240,...,0.909259,584,10,10840,4600,7,110,-0.011667,0.075590,test 仅用于最终观察；不允许根据该表反向修改候选组或阈值
3,median_all_selected_missing_indicators_top30,0.30,0.430804,0.965,0.595679,0.773237,0.860375,255,7,6050,...,0.908459,293,23,14430,8380,16,38,-0.026333,0.047276,test 仅用于最终观察；不允许根据该表反向修改候选组或阈值


`baseline_median_all` 的 official test cost 为 `10820`，`structural_all` 为 `9980`。结构特征带来一定泛化收益，但不是巨大提升；当前项目更强调成本敏感流程和业务交付闭环，而不是单纯追求分数。

In [6]:
test_results.to_csv(cfg.metrics_dir / "day14_structural_feature_test_results.csv", index=False)
valid_test_compare.to_csv(cfg.metrics_dir / "day14_structural_feature_valid_test_compare.csv", index=False)
test_predictions.to_csv(cfg.predictions_dir / "day14_structural_feature_test_predictions.csv", index=False)
metadata.to_csv(cfg.tables_dir / "day14_structural_feature_test_metadata.csv", index=False)

## 小结

- final candidate：`median_all_structural_all / structural_all`。
- fixed threshold：`0.18`。
- official test：FP=`398`，FN=`12`，total_cost=`9980`。
- Day6 `8640` 是测试集回溯阈值观察，不作为 final result。
- Day16 tuned cost=`11260`，Day18 ensemble 未稳定减少 FN，因此最终保留 Day14 structural_all。